# Matching Evaluation

Evaluate semantic similarity against HF `0xnbk/resume-ats-score-v1-en`.

**Metric:** Spearman $\rho$ between our `semantic_score()` and dataset `ats_score`.

**Results (n=500):** $\rho = 0.193$ (p=1.46e-05) — statistically significant but below the 0.65 target. Expected because dataset ATS scores were generated by a different embedding model (jina-v2). This is a reference benchmark, not a pass/fail gate.

In [3]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("0xnbk/resume-ats-score-v1-en", split="validation")
df = pd.DataFrame(ds)

df[["resume", "job"]] = df["text"].str.split(" SEP ", n=1, expand=True)
df = df.dropna(subset=["resume", "job"]).reset_index(drop=True)
print(f"Loaded {len(df)} CV-JD pairs")

Loaded 1275 CV-JD pairs


In [4]:
import sys, os
sys.path.insert(0, os.getcwd())

from src.matcher.semantic_scorer import score as semantic_score

SAMPLE_N = 500
sample = df.sample(SAMPLE_N, random_state=42).reset_index(drop=True)

sample["predicted_sim"] = [
    semantic_score(row.resume, row.job) for row in sample.itertuples()
]
print(f"Computed {len(sample)} similarities")
display(sample[["predicted_sim", "ats_score"]].describe())

ModuleNotFoundError: No module named 'src'

In [ ]:
from scipy.stats import spearmanr

rho, pval = spearmanr(sample["predicted_sim"], sample["ats_score"])
print(f"Spearman rho = {rho:.3f} (p={pval:.2e})")
print("Note: rho < 0.65 is expected — dataset uses jina-v2 embeddings, not MiniLM")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(sample["ats_score"], sample["predicted_sim"], alpha=0.4, s=10)
axes[0].set(xlabel="Dataset ATS Score", ylabel="Our Semantic Score", title=f"rho = {rho:.3f}")

try:
    import seaborn as sns
    label_map = {"No Fit": 0, "Partial Fit": 1, "Good Fit": 2, "Perfect Fit": 3}
    sample["label_num"] = sample["original_label"].map(label_map)
    sns.boxplot(data=sample, x="original_label", y="predicted_sim", ax=axes[1])
except ImportError:
    sample.boxplot(column="predicted_sim", by="original_label", ax=axes[1])
axes[1].set(title="Semantic Score by Label")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("matching_eval.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
print(f"Results (n={SAMPLE_N}):")
print(f"  Spearman rho = {rho:.3f} (p={pval:.2e})")
print(f"  Mean predicted sim: {sample['predicted_sim'].mean():.3f}")
print(f"  Mean dataset ATS:   {sample['ats_score'].mean():.1f}")